In [1]:
import os
import shutil
import numpy as np
import random

In [ ]:
import os
import shutil
import numpy as np
from google.colab import drive
drive.mount('/content/gdrive')
os.chdir('/content/gdrive/MyDrive/PQHIDE')

In [6]:
def modifier_bits(chaine, n, nb_bits_a_modifier):
    # Convertir la chaîne en une liste de bits
    bits = list(chaine)

    # Vérifier que la chaîne a au moins n bits
    if len(bits) < n:
        raise ValueError("La chaîne de caractères est plus courte que n bits.")

    # Choisir aléatoirement les positions des bits à modifier
    positions = random.sample(range(n), nb_bits_a_modifier)

    # Inverser les bits aux positions choisies
    for pos in positions:
        bits[pos] = '1' if bits[pos] == '0' else '1'

    # Convertir la liste de bits en une chaîne de caractères
    nouvelle_chaine = ''.join(bits)

    return nouvelle_chaine

In [9]:
import hashlib

import numpy as np

from utils import gaussjordan

from LinearCode import LinearCode



class QC_MDPC(LinearCode):
    """
    Quasi-Cyclic LDPC code representation (extends LinearCode)

    ...

    Methods
    -------
    from_params(n, p, w)
        Init QC-LDPC by length, circulant size and code weight

    _get_circulant_block(polynom)
        Get circulant (p, p) for given vector of size p

    """

    def __init__(self, G, H):
        super().__init__(G, H)

    @classmethod
    def from_params(cls, n, p, w):
        assert n % p == 0, "p must be delimeter of n"

        n0 = n // p
        assert w > 2*n0, "not enough code weight"

        fine = False

        while not fine:
            blocks = []
            inverse_block = None
            inverse_block_position = None

            vector = [1 for _ in range(w)] + [0 for _  in range(n - w)]
            vector = np.array(vector, dtype=int)
            np.random.shuffle(vector)

            for i in range(n0):
                circ = vector[i*p:(i+1)*p]

                if sum(circ) < 2:
                    inverse_block = None
                    break

                block = QC_MDPC._get_circulant_block(circ)
                blocks.append(block)

                A, P = gaussjordan(block, True)
                A = np.array(A, dtype=int)
                P = np.array(P, dtype=int)

                if (A == np.eye(p, dtype=int)).all():
                    inverse_block_position = i
                    inverse_block = P

            # continue only if inverse circulant found
            fine = True if inverse_block is not None else False

        # put inverse block on last position
        blocks[inverse_block_position], blocks[n0-1] = blocks[n0-1], blocks[inverse_block_position]
        H = np.concatenate(blocks, axis=1)

        for i in range(n0):
            blocks[i] = blocks[i] @ inverse_block % 2
            blocks[i] = blocks[i].T

        Ht = np.concatenate(blocks[:n0-1], axis=0)
        G = np.concatenate((np.eye(Ht.shape[0], dtype=int), Ht), axis=1)

        assert (G @ H.T % 2 == 0).all(), "G is not correspond to H"

        return cls(G, H)

    @staticmethod
    def _get_circulant_block(polynom):
        N = len(polynom)
        block = np.empty((N, N), dtype=int)

        for i in range(N):
            block[i] = np.roll(polynom, i)

        return block

In [ ]:
i = 0
negatif = 0
positif = 0

while i < 1000 :
    #BIKE-3 parameters level 1
    n = 22054
    p = 11027
    w = 134
    errors_num = 154
    qc_mdpc = QC_MDPC.from_params(n, p, w)

    word = np.random.randint(2, size=qc_mdpc.getG().shape[0])
    #print("word = ",word)
    #print(len(word))

    encoded = qc_mdpc.encode(word)
    #print("encoded = ",encoded)
    #print(len(encoded))

    # error vector size n with t or less errors
    e = [1 for _ in range(errors_num)] + [0 for _  in range(n - errors_num)]
    e = np.array(e, dtype=int)
    np.random.shuffle(e)
    #print("error = ",e)
    #print(len(e))

    corrupted = (encoded + e) % 2
    #print("corrupted = ",corrupted)
    #print(len(corrupted))

    #print(''.join(map(lambda x: str(x), corrupted)))
    corrupted = ''.join(map(lambda x: str(x), corrupted))

    n = len(corrupted)
    factor = int(n*0.02)
    chaine_modifiee = modifier_bits(str(corrupted), n, factor)
    #print("Chaîne originale :", corrupted)
    #print("Chaîne modifiée accidentellement :", chaine_modifiee)
    #print(len(chaine_modifiee))

    chaine_modifiee = [int(chaine_modifiee[i]) for i in range(len(chaine_modifiee))]
    chaine_modifiee = np.array(chaine_modifiee, dtype=int)
    #print(chaine_modifiee)
    #liste_modifiee = list(map(int, chaine_modifiee.split()))
    #chaine_modifiee = np.array(chaine_modifiee, dtype=int)
    #print(type(encoded))

    decoded = qc_mdpc.decode(np.copy(chaine_modifiee))
    decoded = qc_mdpc.get_message(decoded)
    #print(decoded)
    #print(len(decoded))

    #print(word)
    #print(len(word))

    try:
        assert (decoded == word).all()
    except AssertionError:
        #print("The secret message must be reforwarded")
        negatif = negatif + 1
    else:
        #print("The secret message has been correctly forwarded")
        positif = positif + 1
    i = i + 1


print("le nombre de negatif est ", negatif)
print("le nombre de positif est ", positif)